# Stable Production — LR-TFIDF + 5-Fold CV

Production settings from `configs/stable_training.yaml`:
- **TF-IDF:** `max_features=800`, bigrams, `sublinear_tf`
- **LR:** `C=0.05` with grid search until train–test gap < 5 pp
- **Augmentation:** toxic-only back-translation (EN→ES→EN) + cosine dedup
- **Evaluation:** stratified 5-fold CV on the train+val pool

Run the full pipeline from repo root:
```bash
uv sync --extra hf --extra train
uv run python -m src.pipeline.run_stable_pipeline
```

## 0. Setup

In [1]:
import json
from pathlib import Path

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

cfg = yaml.safe_load(open(PROJECT_ROOT / "configs" / "stable_training.yaml"))
reports_dir = PROJECT_ROOT / "reports" / "stable"
runs = sorted(reports_dir.glob("stable_run_*.json"))
assert runs, "No stable_run_*.json — run the pipeline first"
latest = runs[-1]
metrics = json.loads(latest.read_text())
run_id = metrics["run_id"]
print(f"Loaded: {latest.name} (run_id={run_id})")

Loaded: stable_run_20260524_190417.json (run_id=20260524_190417)


## 1. Augmentation summary

In [2]:
aug = metrics.get("augmentation", {})
pd.Series(aug)

enabled                          True
strategy             back_translation
train_size_before                 677
train_size_after                  877
added_samples                     200
dtype: object

## 2. LR gap search (holdout test)

In [3]:
lr = metrics["logistic_regression"]
gap_search = metrics.get("lr_gap_search", {})

rows = [
    {"metric": "F1 weighted (test)", "value": lr["f1_weighted"]},
    {"metric": "F1 weighted (train, orig)", "value": lr["f1_train"]},
    {"metric": "Train–test gap (pp)", "value": lr["train_test_gap_pp"]},
    {"metric": "ROC-AUC (test)", "value": lr["roc_auc"]},
    {"metric": "Chosen C", "value": lr.get("C", gap_search.get("C"))},
    {"metric": "max_features", "value": lr.get("max_features", gap_search.get("max_features"))},
    {"metric": "Gap OK (<5pp)", "value": lr.get("gap_ok", gap_search.get("gap_ok"))},
]
display(pd.DataFrame(rows))

,metric,value
0,F1 weighted (test),0.6546
1,"F1 weighted (train, orig)",0.7721
2,Train–test gap (pp),11.74
3,ROC-AUC (test),0.7312
4,Chosen C,0.005
5,max_features,800
6,Gap OK (<5pp),False


## 3. Stratified 5-fold CV (LR)

In [4]:
cv = metrics["cv_logistic_regression"]
print(
    f"F1: {cv['f1_mean']} ± {cv['f1_std']}  |  "
    f"fold gap max: {cv['gap_max']*100:.2f} pp  |  "
    f"stable: {cv['stable_across_folds']}"
)
fold_df = pd.DataFrame(cv["folds"])
fold_df[["fold", "f1_weighted", "train_val_gap_pp", "roc_auc"]]

F1: 0.6636 ± 0.0223  |  fold gap max: 14.66 pp  |  stable: False


,fold,f1_weighted,train_val_gap_pp,roc_auc
0,0,0.680685,11.05,0.7239
1,1,0.650439,12.59,0.7341
2,2,0.697292,7.12,0.7277
3,3,0.653930,13.11,0.6728
4,4,0.635539,14.66,0.6856


## Conclusion

This notebook summarizes **LR-TFIDF** from the latest stable production run.
Check `reports/stable/integrated_report_{run_id}.md` for the combined LR + DistilBERT + ensemble report.
The 5-fold CV **F1 std** measures stability across data segments; the **train–val gap** per fold tracks overfitting within each split.
Target rubric: |train − test| < 5 pp and test F1 > 0.80 — tune `logistic_regression.gap_search.param_grid` if gaps remain high.